<table>
    <tr>
      <td>Minería de datos y paradigma BigData - Facultad de Informática - UCM
      </td>
      <td>
      <img src="https://biblioteca.ucm.es/data/cont/media/www/pag-88746//escudo.jpg" width=50/>
      </td>
     </tr>
</table>

# Práctica 8: Análisis de tráfico de red con PySpark 
### Pablo C. Cañizares

En esta práctica aprenderás a trabajar con **PySpark** para analizar un dataset real de tráfico de red con flujos benignos y ataques. El notebook cubre dos bloques:

1. **Parte I — DataFrames**: carga, limpieza, filtrado, agregación, joins y patrones reales de detección.
2. **Parte II — Machine Learning con MLlib**: clasificación multiclase de ataques con Random Forest.

No te olvides de incluir los nombres y puesto de los miembros del grupo.

Nombres:

Puesto:

---

## Dataset: UNSW-NB15

El dataset **UNSW-NB15** contiene flujos de tráfico de red capturados en la universidad UNSW de Sydney, mezclando tráfico benigno con nueve familias de ataque. Cada fila es un flujo de red con ~45 características, una etiqueta binaria y una categoría de ataque.

Las columnas que más usaremos son:

| Columna | ¿Qué es? | Ejemplo |
|---|---|---|
| `srcip` | IP de origen | 175.45.176.0 |
| `dstip` | IP de destino | 149.171.126.16 |
| `dsport` | Puerto destino | 80 |
| `proto` | Protocolo (tcp, udp, icmp…) | tcp |
| `service` | Servicio detectado (http, dns, ftp, `-`…) | http |
| `state` | Estado de la conexión (FIN, CON, INT…) | FIN |
| `dur` | Duración del flujo (segundos) | 0.121 |
| `spkts` / `dpkts` | Paquetes origen→destino y destino→origen | 10 / 8 |
| `sbytes` / `dbytes` | Bytes origen→destino y destino→origen | 1420 / 830 |
| `sload` / `dload` | Bits por segundo en cada sentido | 95000.4 |
| `ct_srv_src` | Nº de conexiones con el mismo servicio desde la misma IP origen | 3 |
| `attack_cat` | Categoría del ataque | `Exploits`, `DoS`, `Generic`, `Reconnaissance`, `Fuzzers`, `Analysis`, `Backdoor`, `Shellcode`, `Worms`, `Normal` |
| `label` | 0 = benigno, 1 = ataque | 1 |

> **Nota:** El dataset tiene ~2,5M de flujos en su versión completa. Para esta práctica se utiliza una de las particiones(UNSW-NB15_1.csv), con aprox. 700.000 filas.

---

## ⚠️ Consideraciones importantes ⚠️
- **Se recuerda que el uso de IA generativa en la resolución de la práctica conlleva el suspenso de la convocatoria**. 
- **Se recuerda que la práctica es por parejas, no se permite la colaboración entre grupos. Puede conllevar el suspenso de la práctica**.
- La realización de la práctica es **obligatorio** llevarla a cabo en los ordenadores del laboratorio.
- **No se permite el uso de dispositivos móviles ni portátiles durante la sesión**.
- **Antes de entregar la práctica es necesario avisar al profesor para que revise la entrega**. En caso contrario la práctica tendrá un máximo de 5 puntos.
- En la entrega se podrán realizar preguntas de la práctica que **ambos** miembros del grupo deben saber contestar. La calificación puede ser distinta para cada miembro del grupo.
- Databricks tiene IA incluida. Desactiva Genie y su autocompletado: View > Developer Settings > Code Editor > Autocomplete as you type (OFF) y Automatic Genie Code Autocomplete (OFF) (esquina inferior derecha cuando se comete un error de sintaxis).

Otras consideraciones:
- **Plataforma**: La práctica está diseñada para su implementación en la plataforma *Databricks Free Edition*. Debes acceder a la página oficial (https://www.databricks.com) y crear una cuenta.
- **Carga en Databricks:** Sube `UNSW-NB15_1.csv` a tu volumen recien creado (si no lo has creado aún: Catalog > New volume), arrastrando el CSV en el volumen creado. Puedes copiar la ruta del volumen generado (hay un botón de copiar).

## Setup

In [0]:
# Importaciones necesarias
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import NumericType

# Comprobamos que Spark está disponible
print(f"PySpark versión: {spark.version}")

# Ruta al CSV — ajusta si lo has subido a otra ubicación
CSV_PATH = "/Volumes/workspace/default/p8/UNSW-NB15_1.csv"

print(f"Dataset Path: {CSV_PATH}")

PySpark versión: 4.1.0
Dataset Path: /Volumes/workspace/default/p8/UNSW-NB15_1.csv


---

# Parte I: Análisis con DataFrames

Un **DataFrame** en Spark es como una tabla de base de datos distribuida: tiene columnas con nombre y tipo, y Spark optimiza automáticamente las operaciones gracias a su motor interno llamado **Catalyst**.

Vamos a aprender las operaciones fundamentales: cargar datos, limpiar, filtrar, agrupar, transformar y combinar.

---

### Ejercicio 1: Cargar el CSV y medir el desbalance de clases (0,25 puntos)

Carga el fichero CSV como DataFrame y responde:
- ¿Cuántas filas tiene el dataset?
- ¿Cuántas columnas?
- ¿Cuántos flujos son benignos (`label = 0`) y cuántos son ataques (`label = 1`)?

In [0]:
#solución
# Cargamos el CSV con cabecera y detección automática de tipos
columnas = [
    "srcip","sport","dstip","dsport","proto","state","dur","sbytes","dbytes",
    "sttl","dttl","sloss","dloss","service","sload","dload","spkts","dpkts",
    "swin","dwin","stcpb","dtcpb","smeansz","dmeansz","trans_depth","res_bdy_len",
    "sjit","djit","stime","ltime","sintpkt","dintpkt","tcprtt","synack","ackdat",
    "is_sm_ips_ports","ct_state_ttl","ct_flw_http_mthd","is_ftp_login","ct_ftp_cmd",
    "ct_srv_src","ct_srv_dst","ct_dst_ltm","ct_src_ltm","ct_src_dport_ltm",
    "ct_dst_sport_ltm","ct_dst_src_ltm","attack_cat","label"
]

df = (spark.read
    .option("header", "false")
    .option("inferSchema", "true")
    .csv(CSV_PATH)
    .toDF(*columnas)
)

df_benigno = df.filter(df.label==0)
df_maligno = df.filter(df.label==1)

# Número de filas y columnas
print(f"Filas:  ", df.count())
print(f"Columnas: ", len(df.columns))
print(f"Benignos:    ", df_benigno.count())
print(f"Malignos:    ", df_maligno.count())

Filas:   700001
Columnas:  49
Benignos:     677786
Malignos:     22215


---

### Ejercicio 2: Investigar y tratar los valores nulos (1,5 puntos)

Al cargar el dataset hemos detectado que hay una columna con muchos nulos. Tu tarea es:

- a) **Contar los nulos por columna**. Se te proporciona una parte de solución en el dataframe `nulos_por_col`, pero es poco legible. Para ver fácilmente la columna problemática, solo imprime las columnas que tengan más de 10 datos nulos. PD: Para obtener el valor de una columna, basta con utilizar `.first()[0]`.
  
- b) **Averiguar por qué** la columna problemática tiene tantos nulos. Para ello, agrupa por `label` y comprueba si los nulos se concentran en una clase concreta, utiliza el nombre de la columna sospechosa, en combinacion con `F.col("columna_sospechosa").isNull()` para ver si esa agrupación tiene elementos nulos.

- c) **Imputar los nulos** con el valor `"Normal"` usando `fillna()`. Ten en cuenta que `df.fillna(valor, subset=["col"])` reemplaza los nulos de una columna por el valor indicado.

  
- d) **Comprobar** que ya no quedan nulos en esa columna y que ahora tienes `"Normal"` como una categoría más al hacer `groupBy`.

Llama al DataFrame resultante `df_limpio`. Lo usaremos en los ejercicios siguientes.

> **Concepto clave:** `df.fillna(valor, subset=["col"])` reemplaza los nulos de una columna por el valor indicado. Es una operación muy común en preprocesado: no siempre un nulo significa "dato perdido", a veces significa "categoría por defecto".

In [0]:
# solución
# Conteo de nulos por columna (te lo damos hecho)
nulos_por_col = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns
])

# a) Detectar valores con nulos > 10
...

for x in range(len(nulos_por_col.columns)):
    if (nulos_por_col.first()[x] > 10):
        print(nulos_por_col.first()[x])
        print(nulos_por_col[x])

# b) Investigamos dónde se concentran los nulos de las columnas detectadas
...
nulos_benignos = df_benigno.filter(F.col("attack_cat").isNull())
print("Nulos benignos: ", nulos_benignos.count())

nulos_malignos = df_maligno.filter(F.col("attack_cat").isNull())
print("Nulos malignos: ",nulos_malignos.count())



# c) Imputamos los nulos con "Normal"
...
df_limpio = df.fillna("Normal", subset=["attack_cat"])

# d) Comprobación
nulos_por_col_limpio = df_limpio.select([
    F.sum(F.col(d).isNull().cast("int")).alias(d) for d in df_limpio.columns
])
print("Nulos restantes:", nulos_por_col_limpio )

display(df_limpio.groupBy("attack_cat").count())

677786
Column<'attack_cat'>
Nulos benignos:  677786
Nulos malignos:  0
Nulos restantes: DataFrame[srcip: bigint, sport: bigint, dstip: bigint, dsport: bigint, proto: bigint, state: bigint, dur: bigint, sbytes: bigint, dbytes: bigint, sttl: bigint, dttl: bigint, sloss: bigint, dloss: bigint, service: bigint, sload: bigint, dload: bigint, spkts: bigint, dpkts: bigint, swin: bigint, dwin: bigint, stcpb: bigint, dtcpb: bigint, smeansz: bigint, dmeansz: bigint, trans_depth: bigint, res_bdy_len: bigint, sjit: bigint, djit: bigint, stime: bigint, ltime: bigint, sintpkt: bigint, dintpkt: bigint, tcprtt: bigint, synack: bigint, ackdat: bigint, is_sm_ips_ports: bigint, ct_state_ttl: bigint, ct_flw_http_mthd: bigint, is_ftp_login: bigint, ct_ftp_cmd: bigint, ct_srv_src: bigint, ct_srv_dst: bigint, ct_dst_ltm: bigint, ct_src_ltm: bigint, ct_src_dport_ltm: bigint, ct_dst_sport_ltm: bigint, ct_dst_src_ltm: bigint, attack_cat: bigint, label: bigint]


attack_cat,count
Reconnaissance,1759
Analysis,526
Normal,677786
Worms,24
Exploits,5409
Generic,7522
Shellcode,223
Fuzzers,5051
DoS,1167
Backdoors,534


---

### Ejercicio 3: ¿En qué se diferencian los flujos benignos de los ataques? (1 punto)

Usa `groupBy("label").agg(...)` sobre `df_limpio` para calcular, para cada clase (benigno vs ataque), estas cuatro estadísticas:

- **Media** de `dur`
- **Media** de `sbytes`
- **Desviación típica** de `sbytes`
- **Máximo** de `dpkts`

Renombra las columnas del resultado a algo legible (`dur_medio`, `sbytes_medio`, `sbytes_std`, `dpkts_max`).

> **Concepto clave:** `.agg()` acepta varias funciones a la vez y `.alias("nombre")` cambia el nombre de la columna resultante. Para las medias, desviacion etc, utiliza las funciones auxiliares de spark con `F`.

In [0]:
#solución
# Estadísticas por clase (label)
stats_clase = df_limpio.groupBy("label").agg(F.mean("dur").alias("dur_medio"), F.mean("sbytes").alias("sbytes_medio"), F.stddev("sbytes").alias("sbytes_std"), F.max("dpkts").alias("dpkts_max"))

display(stats_clase)


label,dur_medio,sbytes_medio,sbytes_std,dpkts_max
0,0.8531379930700865,4745.577971513132,13176.527154573881,1278
1,1.1762109027233905,13066.138329957235,241163.00456241454,10970


---

### Ejercicio 4: Filtro compuesto con rango y pertenencia a lista (0,75 puntos)

Sobre `df_limpio`, cuenta cuántos flujos cumplen **todas** estas condiciones a la vez:

- `proto` es `"tcp"`
- `state` es uno de: `"FIN"`, `"CON"`
- `sbytes` está entre **200 y 5000** (ambos incluidos)
- `service` es `"http"` o `"-"` (el guion indica servicio no identificado)

> **Conceptos clave:** `F.col("x").isin(["a","b"])` comprueba pertenencia a una lista, y `F.col("x").between(a,b)` hace lo propio con un rango. Combínalos con `&` y paréntesis.

In [0]:
#solución
# Filtra y cuenta
flujos_filtrados = df_limpio ...
print(f"Flujos que cumplen todas las condiciones: {flujos_filtrados.count()}")


---

### Ejercicio 5: Servicios exclusivos de ataques (1 punto)

Queremos la lista de **servicios** (`service`) que aparecen en flujos de ataque (`label = 1`) **pero no** en flujos benignos (`label = 0`). Devuélvelos ordenados alfabéticamente.

> **Pista:** puedes hacerlo con dos `DataFrames` (uno por cada clase) y la operación `subtract()` o `exceptAll()`, que resta los elementos comunes.

In [0]:
#solución
# Servicios presentes en ataques
servicios_ataque = df_limpio ...

# Servicios presentes en benignos
servicios_benignos = df_limpio ...

# Solo en ataques (resta los comunes)
solo_ataques = ...

display(solo_ataques)

---

### Ejercicio 6: Top-10 flujos más voluminosos por categoría (0,5 puntos)

Sobre `df_limpio`, filtra solo los flujos que sean ataques (`label = 1`) y muestra los **10 con mayor `sbytes`**. Selecciona para visualizar únicamente las columnas: `srcip`, `dstip`, `proto`, `service`, `sbytes`, `attack_cat`.

>Pista: Para ordenar de manera descendente combina `orderBy` y `F.col("").desc()`.

In [0]:
#solución
# Top-10 ataques más voluminosos
top10_ataques = df_limpio ... 

display(top10_ataques)


---

### Ejercicio 7: Intensidad del flujo + cruce por categoría de ataque (1 punto)

Crea una nueva columna `intensidad` en `df_limpio` que clasifique cada flujo según `sbytes`:

| Condición | Intensidad |
|---|---|
| `sbytes` > 100 000 | ALTA |
| 1 000 ≤ `sbytes` ≤ 100 000 | MEDIA |
| `sbytes` < 1 000 | BAJA |

Después, muestra una tabla con el **conteo de flujos por `intensidad` y `attack_cat`** (un `groupBy` de dos columnas), ordenada de mayor a menor conteo.

> **Concepto clave:** `F.when(cond1, val1).when(cond2, val2).otherwise(val3)` funciona como `if-elif-else` sobre columnas. Y `groupBy` admite varias columnas.

In [0]:
#solución
# Crea la columna intensidad
df_intensidad =  ...

# Agrupa por intensidad y attack_cat y cuenta
resumen = ...

display(resumen)


---

### Ejercicio 8: Enriquecer el dataset con táctica MITRE y severidad (join) (1 punto)

En un entorno real, los analistas de seguridad no solo quieren saber si un evento es normal o malicioso, sino también **qué objetivo persigue el atacante** y **qué nivel de gravedad tiene**. Para ello, un equipo de seguridad o **SOC** (*Security Operations Center*) nos proporciona, para cada categoría de ataque, dos datos adicionales:

- una **táctica MITRE ATT&CK**, que describe el objetivo general del atacante dentro de una taxonomía estándar de ciberseguridad,
- y una **severidad base**, que indica la criticidad inicial del ataque.

Queremos añadir esa información a nuestro dataset mediante un **join**. En otras palabras, vamos a enriquecer el dataset original con nuevas columnas que aporten más contexto sobre cada tipo de ataque.

In [0]:
#solución
# Tabla de severidad y táctica MITRE por categoría de ataque
severidad_data = [
    ("Generic",         "Impact",            "MEDIA"),
    ("Exploits",        "Execution",         "CRITICA"),
    ("Fuzzers",         "Reconnaissance",    "BAJA"),
    ("DoS",             "Impact",            "CRITICA"),
    ("Reconnaissance",  "Reconnaissance",    "MEDIA"),
    ("Analysis",        "Discovery",         "MEDIA"),
    ("Backdoor",        "Persistence",       "CRITICA"),
    ("Shellcode",       "Execution",         "ALTA"),
    ("Worms",           "Lateral Movement",  "ALTA"),
]

df_severidad = spark.createDataFrame(severidad_data, ["attack_cat", "tactica_mitre", "severidad"])

# Join: añade la severidad y la táctica al dataset principal
df_enriquecido = ...

# Comprobación: cuenta flujos por severidad (ojo con los nulos de los `Normal`)
#display(df_enriquecido.groupBy("severidad").count().orderBy("severidad"))
df_enriquecido.groupBy("severidad").count().orderBy("severidad").show()


---

### Ejercicio 9: Detección de port scanning (1,5 puntos)

El **port scanning** es una técnica de reconocimiento en la que un atacante lanza muchas conexiones cortas contra distintos puertos de una víctima para descubrir servicios abiertos. Las características típicas son:

- Una misma IP de origen genera **muchos** flujos.
- Los flujos son **muy cortos** en duración.
- Se dirigen a **muchos puertos destino distintos**.

**Tu tarea:** a partir de `df_limpio`, identifica las IPs de origen sospechosas de estar escaneando puertos. Los criterios concretos son:

1. Agrupa por `srcip`.
2. Para cada `srcip` calcula:
   - número total de flujos (`n_flujos`),
   - duración media del flujo (`dur_media`),
   - número de puertos destino distintos (`n_puertos`).
3. Filtra las IPs que cumplan las tres condiciones a la vez:
   - `n_flujos > 50`
   - `dur_media < 1`
   - `n_puertos > 3`
4. Ordena el resultado por `n_flujos` de mayor a menor.

> **Conceptos clave:** `F.countDistinct("columna")` cuenta valores únicos dentro de una agrupación. Puedes aplicar `.filter()` **después** de `.agg()` para filtrar sobre las columnas ya calculadas.

In [0]:
#solución
# Detección de port scanning
sospechosos = ...

display(sospechosos)


---

# Parte II: Clasificación de ataques con MLlib

Ahora vamos a entrenar un modelo de **Machine Learning** para que clasifique automáticamente la categoría de ataque (`attack_cat`) a partir de las características del flujo de red.

**¿Qué modelo usamos?** Un **Random Forest** (bosque aleatorio): un conjunto de muchos árboles de decisión que "votan" la clase final. Funciona muy bien con datos tabulares y es fácil de interpretar.

**¿Qué pasos seguiremos?**
1. Seleccionar las columnas que usará el modelo (features)
2. Transformar textos a números (los modelos solo entienden números)
3. Dividir en datos de entrenamiento y de test
4. Entrenar y evaluar

---

## Selección de features

In [0]:
from pyspark.sql import functions as F

# Columnas numéricas candidatas
feature_cols_numeric = [
    "dur", "spkts", "dpkts", "sbytes", "dbytes",
    "sload", "dload", "sttl", "dttl",
    "smeansz", "dmeansz",
    "ct_srv_src", "ct_srv_dst",
    "ct_dst_ltm", "ct_src_ltm"
]

# Columnas categóricas
feature_cols_categorical = ["proto", "service", "state"]

# Objetivo
target_col = "attack_cat"

# Columnas solicitadas
requested_cols = feature_cols_numeric + feature_cols_categorical + [target_col]

# Selección segura (solo las que existan realmente)
cols_existentes = [c for c in requested_cols if c in df_limpio.columns]
df_ml = df_limpio.select(*cols_existentes)

# Eliminar filas con nulos en cualquiera de las columnas seleccionadas
# y flujos sin categoría asignada
df_ml = (df_ml
    .na.drop()
    .filter((F.col(target_col).isNotNull()) & (F.col(target_col) != ""))
)

print(f"Registros tras limpieza: {df_ml.count()}")
#display(df_ml.groupBy(target_col).count().orderBy(F.desc("count")))
df_ml.groupBy(target_col).count().orderBy(F.desc("count")).show()


---

## Ejercicio 10: Construir el Pipeline (1,5 puntos)

Un **Pipeline** encadena varios pasos de transformación y el modelo en una sola secuencia. Esto tiene dos ventajas: evita errores y facilita reproducir el proceso.

Nuestro pipeline:
1. `StringIndexer` → convierte texto a números (`proto`, `service`, `state`, `attack_cat`). Crea un `StringIndexer` por cada columna categórica.
2. `VectorAssembler` → junta todas las features en un único vector llamado `features`.
3. `RandomForestClassifier` → el modelo que aprende a clasificar.

In [0]:
#solución
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

# Un StringIndexer por cada columna categórica. La columna de salida debe llamarse <col>_idx
indexers = [
    StringIndexer(...)
    for c in feature_cols_categorical
]

# Indexer para la etiqueta attack_cat. La columna de salida debe llamarse label
label_indexer = ...

# Unir todas las features en un vector. A la salida del VectorAssembler llámalo features
todas_las_features = ...
assembler = VectorAssembler(...)

# Modelo Random Forest
rf = RandomForestClassifier(...)

# Encadenamos todo en un Pipeline
pipeline = Pipeline(stages=indexers + [label_indexer, assembler, rf])

print("Pipeline construido correctamente.")


---

## Dividir los datos y entrenar

In [0]:
# 80% para entrenar, 20% para evaluar. Usa randomSplit con semilla 42
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

print(f"Datos de entrenamiento: {train_df.count()}")
print(f"Datos de test:          {test_df.count()}")


---

## Predicciones y evaluación

In [0]:
# Para evitar problemas de memoria, entrenamos con solo el 10% del train
train_df_small = train_df.sample(fraction=0.1, seed=42)
print(f"Entrenando con {train_df_small.count()} registros (10% del train)")

# Entrenamos el pipeline con train_df_small y generamos predicciones sobre test_df
modelo = pipeline.fit(train_df_small)
predicciones = modelo.transform(test_df)

# Veamos algunas predicciones
#display(predicciones.select(target_col, "label", "prediction", "probability").limit(10))
predicciones.select(target_col, "label", "prediction", "probability").limit(10).show()


In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Calculamos las métricas principales
metricas = {}
for nombre_metrica in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
    evaluator = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName=nombre_metrica
    )
    metricas[nombre_metrica] = evaluator.evaluate(predicciones)

print("Resultados del modelo: ")
print(f"  > Accuracy:            {metricas['accuracy']:.4f}")
print(f"  > F1 Score:            {metricas['f1']:.4f}")
print(f"  > Precision (media):   {metricas['weightedPrecision']:.4f}")
print(f"  > Recall (media):      {metricas['weightedRecall']:.4f}")


---

***
[![Licencia Creative Commons](https://i.creativecommons.org/l/by/4.0/88x31.png)](https://creativecommons.org/licenses/by/4.0/)

This work has been created by Pablo C. Cañizares. This work is licensed under a [Creative Commons Attribution 4.0 International License](https://creativecommons.org/licenses/by/4.0/).
